# Un petit réseau de neurones avec NumPy

**Objectifs.** Calculer une forward pass, relier les dimensions, calculer une loss binaire et implémenter la rétropropagation avec la règle de chaîne. Durée : 90 min.

Le XOR est un petit problème synthétique non linéaire : une droite ne sépare pas ses classes correctement. Le réseau suivant est une démonstration pédagogique, pas une architecture à réutiliser en production.

## 1. Formes et calcul avant

Pour `X (n, 2)`, `W1 (2, h)`, `b1 (h,)`, on a `Z1 = X @ W1 + b1` de forme `(n,h)`. Après tanh, `A1` conserve la forme. `W2 (h,1)` donne un logit `(n,1)`, converti en probabilité par sigmoid.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([[0], [1], [1], [0]], dtype=float)
rng = np.random.default_rng(3)
hidden_units = 4
W1 = rng.normal(0, 0.6, size=(2, hidden_units)); b1 = np.zeros((1, hidden_units))
W2 = rng.normal(0, 0.6, size=(hidden_units, 1)); b2 = np.zeros((1, 1))
print('X:', X.shape, 'W1:', W1.shape, 'W2:', W2.shape)

## 2. Forward pass et loss

La sigmoid transforme un logit en nombre de 0 à 1. La binary cross-entropy pénalise les probabilités incompatibles avec le label. On clippe uniquement dans le logarithme pour éviter `log(0)` en virgule flottante.

In [ ]:
def sigmoid(value):
    return 1 / (1 + np.exp(-value))

def forward(features, w1, b1, w2, b2):
    z1 = features @ w1 + b1
    a1 = np.tanh(z1)
    logits = a1 @ w2 + b2
    probabilities = sigmoid(logits)
    return z1, a1, logits, probabilities

def binary_cross_entropy(target, probability):
    clipped = np.clip(probability, 1e-8, 1 - 1e-8)
    return -np.mean(target * np.log(clipped) + (1 - target) * np.log(1 - clipped))

_, hidden, logits, probability = forward(X, W1, b1, W2, b2)
print('hidden/output:', hidden.shape, probability.shape)
print('initial BCE:', round(float(binary_cross_entropy(y, probability)), 4))

## 3. Rétropropagation

Avec sigmoid + cross-entropy, le gradient du logit vaut `(probability - target) / n`. Les gradients de `W2,b2` se calculent ensuite. La règle de chaîne transmet le signal à la couche précédente ; pour tanh, `d tanh(z)/dz = 1 - tanh(z)^2`.

On applique les mises à jour avec gradient descent batch.

In [ ]:
learning_rate = 0.7
loss_history = []
for epoch in range(5_000):
    z1, a1, _, p = forward(X, W1, b1, W2, b2)
    loss_history.append(float(binary_cross_entropy(y, p)))
    delta2 = (p - y) / len(X)
    grad_w2 = a1.T @ delta2
    grad_b2 = delta2.sum(axis=0, keepdims=True)
    delta1 = (delta2 @ W2.T) * (1 - a1 ** 2)
    grad_w1 = X.T @ delta1
    grad_b1 = delta1.sum(axis=0, keepdims=True)
    W2 -= learning_rate * grad_w2; b2 -= learning_rate * grad_b2
    W1 -= learning_rate * grad_w1; b1 -= learning_rate * grad_b1
print('final BCE:', round(loss_history[-1], 4))
print('probabilities:', forward(X, W1, b1, W2, b2)[-1].round(3).ravel())

## 4. Visualiser et interpréter

La perte doit décroître pour ces paramètres et ce problème. Le résultat dépend de l’initialisation ; entraînez plusieurs graines pour mesurer cette variabilité. XOR n’est pas un benchmark représentatif.

In [ ]:
plt.plot(loss_history)
plt.xlabel('Epoch'); plt.ylabel('Binary cross-entropy')
plt.title('Loss du MLP NumPy'); plt.grid(alpha=0.2); plt.show()
print('prédictions binaires:', (forward(X, W1, b1, W2, b2)[-1] >= 0.5).astype(int).ravel())

## 5. Ce que font les frameworks

Autograd construit les dérivées à partir des opérations de `forward`. Un framework gère aussi calcul GPU, batchs, initialisations, optimizers et états ; la dérivée sous-jacente reste la règle de chaîne. La boucle explicite de ce notebook est gardée pour comprendre ce que l’outil automatise.

**Limites :** paramètres non convexes, sensibilité aux graines et à l’échelle, loss qui peut diverger si le pas est trop grand. Dropout et batch normalization sont des notions à étudier après la boucle simple.

## Exercice

Ajoutez un point bruité, essayez 2 neurones cachés puis 8 et comparez 5 graines. Rapportez moyenne de la loss finale et variance. Une seule réussite ne suffit pas à conclure sur la robustesse.